# 대규모 모델 직접 배우기: 모델 워터마킹

소개: 이 섹션에서는 언어 모델의 워터마킹을 다룹니다.

> 언어 모델이 생성하는 콘텐츠에 사람이 인식할 수 없지만 알고리즘으로는 감지할 수 있는 "워터마크"를 삽입합니다.

## 이 튜토리얼의 목표

1. 워터마크 삽입: 언어 모델이 콘텐츠를 생성할 때 워터마크 임베딩
2. 워터마크 감지: 주어진 텍스트의 워터마크 강도 측정
3. 워터마크 평가: 워터마크 방법의 감지 성능 평가
4. 워터마크 견고성(Robustness) 평가 (선택사항)


## 사전 준비

### 2.1 X-SIR 코드 저장소 소개

https://github.com/zwhe99/X-SIR

X-SIR 저장소에는 다음 내용의 구현이 포함되어 있습니다.

- 세 가지 텍스트 워터마킹 알고리즘: X-SIR, SIR, KGW
- 두 가지 워터마크 제거 공격 방법: 패러프레이즈(paraphrase)와 번역(translation)

![img](./assets/x-sir.png)

### 2.2 환경 준비

```Shell
git clone https://github.com/zwhe99/X-SIR && cd X-SIR
conda create -n xsir python==3.10.10
conda activate xsir
pip3 install -r requirements.txt
# [선택사항] pip3 install flash-attn==2.3.3
```

> requirements.txt의 버전은 권장 버전이며, 반드시 지켜야 하는 것은 아닙니다.




## 실습 예시

> KGW 알고리즘을 사용하여 언어 모델 생성 콘텐츠에 워터마크를 삽입합니다.

### 3.1 데이터 준비

언어 모델에 입력할 프롬프트(prompt)를 jsonl 파일 형식으로 구성합니다:

```JSON
{"prompt": "Ghost of Emmett Till: Based on Real Life Events "}
{"prompt": "Antique Cambridge Glass Pink Decagon Console Bowl Engraved Gold Highlights\n"}
{"prompt": "2009 > Information And Communication Technology Index statistics - Countries "}
......
```

- 각 줄은 하나의 JSON 객체이며, 최소한 "prompt"라는 키를 포함해야 합니다.
- 이후 내용은 `data/dataset/mc4/mc4.en.jsonl` 파일을 예시로 사용합니다. 이 파일에는 총 500개의 데이터가 포함되어 있으며, 모델 처리 시간이 너무 길다면 데이터를 직접 줄여도 됩니다.

### 3.2 워터마크 삽입

- 모델과 워터마킹 알고리즘을 선택합니다. 여기서는 `baichuan-inc/Baichuan-7B` 모델과 `KGW` 워터마킹 알고리즘을 선택합니다.

  - ```Shell
    MODEL_NAME=baichuan-inc/Baichuan-7B
    MODEL_ABBR=baichuan-7b
    WATERMARK_METHOD_FLAG="--watermark_method kgw"
    ```

- 콘텐츠를 생성하고 워터마크를 삽입합니다.

  - ```Shell
    python3 gen.py \
        --base_model $MODEL_NAME \
        --fp16 \
        --batch_size 32 \
        --input_file data/dataset/mc4/mc4.en.jsonl \
        --output_file gen/$MODEL_ABBR/kgw/mc4.en.mod.jsonl \
        $WATERMARK_METHOD_FLAG
    ```

  - 이 명령어는 모델이 생성한 콘텐츠를 출력 파일에 저장합니다: `gen/$MODEL_ABBR/kgw/mc4.en.mod.jsonl`

  - 출력 파일의 형식은 다음과 같으며, response는 모델의 출력 내용입니다:

    - ```JSON
      {"prompt": "Ghost of Emmett Till: Based on Real Life Events ", "response": ".In August if 1955 African American Emmett Louis Till (21)\nThe second part of The Man From Waco..."}
      {"prompt": "Antique Cambridge Glass Pink Decagon Console Bowl Engraved Gold Highlights\n", "response": "An exceptionally fine decorative antique pink decagonal glass side bowl..."}
      ......
      ```



### 3.3 워터마크 감지

> 워터마크 감지란 주어진 텍스트에 대해 해당 텍스트의 워터마크 강도(z-score)를 계산하는 것입니다.

- **워터마크가 있는** 텍스트의 워터마크 강도 계산

  - ```python
    python3 detect.py \
        --base_model $MODEL_NAME \
        --detect_file gen/$MODEL_ABBR/kgw/mc4.en.mod.jsonl \
        --output_file gen/$MODEL_ABBR/kgw/mc4.en.mod.z_score.jsonl \
        $WATERMARK_METHOD_FLAG
    ```

- **워터마크가 없는** 텍스트의 워터마크 강도 계산

  - ```python
    python3 detect.py \
        --base_model $MODEL_NAME \
        --detect_file data/dataset/mc4/mc4.en.jsonl \
        --output_file gen/$MODEL_ABBR/kgw/mc4.en.hum.z_score.jsonl \
        $WATERMARK_METHOD_FLAG
    ```

- 출력 파일의 형식:

  - ```JSON
    {"z_score": 12.105422509165574, "prompt": "Ghost of Emmett Till: Based on Real Life Events ", "response": "...", "biases": null}
    {"z_score": 12.990684249887122, "prompt": "Antique Cambridge Glass Pink Decagon Console Bowl Engraved Gold Highlights\n", "response": "...", "biases": null}
    {"z_score": 11.455466938203664, "prompt": "2009 > Information And Communication Technology Index statistics - Countries ", "response": "...", "biases": null}
    ......
    ```

- 두 파일의 워터마크 강도 차이를 눈으로 비교해 봅니다.

### 3.4 워터마크 평가
<a name="eval"></a>

- 워터마크 감지의 z-score 파일을 입력으로 받아, 감지 정확도를 계산하고 ROC 곡선을 그립니다.

  - ```Shell
    python3 eval_detection.py \
            --hm_zscore gen/$MODEL_ABBR/kgw/mc4.en.hum.z_score.jsonl \
            --wm_zscore gen/$MODEL_ABBR/kgw/mc4.en.mod.z_score.jsonl \
            --roc_curve roc
    
    AUC: 1.000
    
    TPR@FPR=0.1: 0.998
    TPR@FPR=0.01: 0.998
    
    F1@FPR=0.1: 0.999
    F1@FPR=0.01: 0.999
    ```

![img](./assets/curve.png)


## 워터마크 견고성(Robustness) 평가 (선택사항)

> 워터마크 텍스트에 패러프레이즈(paraphrase) 및 번역(translation) 공격을 가한 후, 감지 효과를 재평가합니다.

### 4.1 사전 준비

gpt-3.5-turbo-1106 모델을 사용하여 워터마크 텍스트에 패러프레이즈 및 번역을 수행합니다. 다른 도구를 선택해도 됩니다.

- OpenAI API 키 설정

  - ```Shell
    export OPENAI_API_KEY=xxxx
    ```

- `attack/const.py`에서 RPM(분당 요청 수)과 TPM(분당 토큰 수)을 수정합니다.

### 4.2 공격 수행 (번역을 예시로)

- 워터마크 텍스트를 한국어(또는 중국어)로 번역

  - ```Shell
    python3 attack/translate.py \
        --input_file gen/$MODEL_ABBR/kgw/mc4.en.mod.jsonl \
        --output_file gen/$MODEL_ABBR/kgw/mc4.en-zh.mod.jsonl \
        --model gpt-3.5-turbo-1106 \
        --src_lang en \
        --tgt_lang zh
    ```

- 재평가

  -  [3.4](#eval) 참조

- 공격 전후 워터마크 성능 변화를 비교합니다.




## 심화 연습

- [X-SIR](https://github.com/zwhe99/X-SIR) 문서를 참고하여 다른 두 가지(X-SIR, SIR) 알고리즘 사용법을 익히고, 다양한 공격 방법 하에서의 성능을 평가해 보세요.